# Industrie et nappes phréatiques dans le Nord

## Introduction

Pour produire 1 kg d'acier, il faut utiliser entre 300 et 600 litres d'eau. L'eau est largement utilisée dans l'industrie pour refroidir les centrales, laver et alimenter les machines... Les usines consomment donc beaucoup d'eau dans leur processus de production.  
Or, les usines ont tendance à s'agglomérer autour de grands sites industriels pour des raisons d'efficacité.  
Nous pouvons donc nous demander quel est l'état des nappes phréatiques autour de ces grands sites industriels.

**Problématique** : dans quelle mesure l'exposition locale aux activités industrielles (ICPE) amplifie-t-elle la vulnérabilité et la baisse de niveau des nappes phréatiques ?

Derrière cette question s'en pose une autre, plus prospective : l'implantation des nouvelles méga-usines (batteries, hydrogène, data centers) se fait-elle de manière aveugle dans des bassins versants déjà condamnés à un épuisement de leurs nappes phréatiques ?

**Démarche.** Le niveau d'une nappe dépend de trois choses : la géographie, la pluie et l'industrie. Après la récupération des données (partie 1) et un premier croisement entre usines et stations (partie 2), on isole l'effet de la géographie (partie 3), puis celui du climat (partie 4), avant de mesurer celui de l'industrie (partie 5).

# 1. Récupération et préparation des données

In [ ]:
import os
import ast
import time

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import requests as rq
import contextily as ctx
from scipy.stats import spearmanr
from matplotlib.colors import TwoSlopeNorm

#AFFICHER vaut True quand on ouvre ce notebook seul, et False quand un autre notebook l'exécute (étape suivante ou rapport) :
#dans ce cas, on fait tous les calculs mais on ne trace aucune figure
AFFICHER = globals().get("AFFICHER", True)

## 1.1 Installations industrielles (ICPE)
On utilise l'API de Géorisques pour récupérer les informations sur les usines.

In [ ]:
#On crée le dossier data qui va stocker toutes nos données :
os.makedirs("../../data", exist_ok=True)

#On récupère la liste des installations industrielles à l'échelle nationale sur le site de l'ICPE
url_icpe = "https://mapsref.brgm.fr/wxs/georisques/georisques_dl?&service=wfs&version=2.0.0&request=getfeature&typename=InstallationsClassees&outputformat=CSVTEXT"

lien_stockage_physique_icpe = "../../data/icpe.csv"

#Si le fichier a déjà été téléchargé, on réutilise la copie locale (le téléchargement est long)
if os.path.exists(lien_stockage_physique_icpe):
    print("Fichier ICPE déjà téléchargé, on réutilise la copie locale.")
else:
    fichier_icpe = rq.get(url_icpe)

    if fichier_icpe.status_code == 200:
        with open(lien_stockage_physique_icpe, "wb") as f:
            f.write(fichier_icpe.content)
        print("Fichier téléchargé.")
    else:
        print("Erreur avec le téléchargement du fichier ICPE.")

In [ ]:
# On récupère donc la liste des installations industrielles
installations_indust = pd.read_csv(lien_stockage_physique_icpe, sep=";", encoding='utf-8')
print("On a ", installations_indust.shape[0], "installations industrielles.", end="")
installations_indust = installations_indust.loc[installations_indust["lib_regime"] != "Non ICPE"]
print(" Dont ", installations_indust.shape[0], "installations industrielles ICPE.")

#On veut garder uniquement les installations industrielles pures (pas l'agriculture et l'élevage par ex)
installations_indust = (installations_indust.loc[
    (installations_indust["industrie"] == 1)
    & (installations_indust["bovins"] == 0)
    & (installations_indust["porcs"] == 0)
    & (installations_indust["volailles"] == 0)
    & (installations_indust["eolienne"] == 0)
    ])
print("On ne garde que les ", installations_indust.shape[0], "installations industrielles pures.")

#On classe les départements qui ont le plus d'installations industrielles pour limiter la recherche (car les données sur Hub'eau sont trop volumineuses pour traiter la France entière)
classement_dpt = (installations_indust.groupby("num_dep")
    .size()
    .sort_values(ascending=False)
    )

In [ ]:
def fig_classement_departements():
    """Nombre d'installations industrielles par département, du plus industriel au moins industriel."""
    plt.figure()
    plt.bar(range(classement_dpt.shape[0]), classement_dpt.values)
    plt.title("Nombre d'installations industrielles dans chaque département")
    plt.xlabel("Rang du département (0 : le département avec le plus d'installations)")
    plt.ylabel("Nombre d'installations industrielles")
    plt.show()

if AFFICHER:
    fig_classement_departements()

On prend par défaut le département qui contient le plus d'installations industrielles pour notre analyse.  
En effet, l'API Hub'eau nous donne un très grand volume de données et nous devons nous limiter dans notre recherche à quelques départements.  
Avec le graphique ci-dessus, on voit que le département 59 est de loin le plus grand centre industriel français.  
Nous nous concentrerons donc sur le département 59 pour la suite de l'analyse.  
La liste `DEPARTEMENTS` ci-dessous permet d'en étudier plusieurs à la fois : toute la suite de l'analyse s'applique à tous les départements de la liste,
et chaque département n'est téléchargé qu'une fois (fichiers `data/..._<département>.csv`).

In [ ]:
#Départements étudiés : par défaut celui qui a le plus d'ICPE (le 59)
#Pour en étudier plusieurs, il suffit de compléter la liste, par exemple : DEPARTEMENTS = ["59", "88", "68"]
DEPARTEMENTS = [str(classement_dpt.index[0])]
print("Départements analysés : ", DEPARTEMENTS)

Mise en forme des données géographiques avec Geopandas

In [ ]:
installations_indust = installations_indust.loc[installations_indust["num_dep"].isin(DEPARTEMENTS)]

#On vérifie qu'il n'y a pas d'installations industrielles sans coordonnées géographiques dans la base de données ICPE des installations industrielles des départements étudiés
nb_position_less = installations_indust.loc[installations_indust["x"].isnull() | installations_indust["y"].isnull() | installations_indust["code_epsg"].isnull()].shape[0]
print(nb_position_less, "installations industrielles n'ont pas de coordonnées géographiques.")
installations_indust = installations_indust.loc[installations_indust["x"].notnull() & installations_indust["y"].notnull() & installations_indust["code_epsg"].notnull()]

#On utilise les coordonnées des installations pour les localiser avec Geopandas
installations_indust["geometry"] = None #Colonne qui va prendre la valeur position Geopandas
codes_epsg_type_coord = installations_indust["code_epsg"].unique()   #On veut que ça marche pour les différents types de référentiels (x,y)
for epsg in codes_epsg_type_coord:
    filtre = installations_indust["code_epsg"] == epsg
    points = gpd.points_from_xy(
        installations_indust.loc[filtre, "x"], 
        installations_indust.loc[filtre, "y"]
    )
    serie_geometrique = gpd.GeoSeries(points, crs=f"EPSG:{int(epsg)}")
    serie_geometrique = serie_geometrique.to_crs("EPSG:2154") #On ramène tout sous le même format
    installations_indust.loc[filtre, "geometry"] = serie_geometrique.values

#On définit un GeoDataFrame pour les études géométriques :
installations_indust = gpd.GeoDataFrame(installations_indust, geometry="geometry", crs="EPSG:2154")

## 1.2 Nappes phréatiques des départements étudiés

L'API Hub'eau ne permet pas d'envoyer toutes les stations d'un coup.  
Encore moins toutes les chroniques d'un coup.  
Donc on ne sélectionne que les départements étudiés (par défaut le 59, où l'on a vu qu'il y avait une grande concentration d'industrie).

In [ ]:
morceaux = []
for dep in DEPARTEMENTS:
    lien_stations = f"../../data/stations_nappes_{dep}.csv"   #Un fichier en cache par département
    if not os.path.exists(lien_stations):
        #On récupère les stations de mesure des nappes phréatiques sur le site Hubeau
        url_stations = "https://hubeau.eaufrance.fr/api/v1/niveaux_nappes/stations"
        params = {"code_departement": dep, "size": 5000, "format": "json"}
        fichier_stations = rq.get(url_stations, params=params)

        if fichier_stations.status_code == 200:
            pd.DataFrame(fichier_stations.json()["data"]).to_csv(lien_stations, encoding='utf-8')
            print("Stations du département", dep, "téléchargées.")
        else:
            print(fichier_stations.status_code)
            print("Erreur avec le téléchargement du fichier stations Hubeau.")
    morceaux.append(pd.read_csv(lien_stations, index_col=0).assign(departement=dep))   #On garde le département étudié de chaque station
stations = pd.concat(morceaux, ignore_index=True)

#Dans le CSV, les listes (masses d'eau, entités BD LISA) sont stockées en texte : on les reconvertit en listes
for colonne in ["codes_bdlisa", "urns_bdlisa", "codes_masse_eau_edl", "noms_masse_eau_edl", "urns_masse_eau_edl"]:
    stations[colonne] = stations[colonne].apply(lambda v: ast.literal_eval(v) if isinstance(v, str) else v)
print("On a ", stations.shape[0], "stations de mesure des nappes phréatiques dans les départements ", DEPARTEMENTS, ".")

Mise en forme des données géographiques avec Geopandas

In [ ]:
#On vérifie qu'il n'y a pas de stations sans coordonnées géographiques
nb_position_less = stations.loc[stations["x"].isnull() | stations["y"].isnull()].shape[0]
print(nb_position_less, "stations n'ont pas de coordonnées géographiques.")
stations = stations.loc[stations["x"].notnull() & stations["y"].notnull()]

#On crée le modèle Geopandas
stations = gpd.GeoDataFrame(
    stations,
    geometry=gpd.points_from_xy(stations['x'], stations['y']),
    crs="EPSG:4326"
).to_crs(epsg=2154) #On ramène tout sous le même format que les installations industrielles

Pour chaque station des départements étudiés, on récupère les séries temporelles de niveau des nappes phréatiques entre janvier 2015 et janvier 2026.

In [ ]:
#On ne peut pas récupérer les mesures des nappes phréatiques pour toutes les stations en même temps.
#Quand on essaye de récupérer toutes les données de toutes les nappes, on a une erreur 414 : la requête URL est trop longue avec tous les code_bss en paramètres.
#Et Hub'eau refuse d'aller au-delà de 20 000 lignes par requête (page x taille de page <= 20 000) : avec plus, il répond une erreur 400.
#Donc on demande les stations par petits paquets, et on recoupe un paquet trop gros en deux (puis la période en deux pour une seule station).
url_chroniques_api = "https://hubeau.eaufrance.fr/api/v1/niveaux_nappes/chroniques"

def requete_hubeau(params, url=url_chroniques_api):
    """Une requête Hub'eau, réessayée en cas de coupure ou de serveur saturé (mais pas en cas de requête refusée)."""
    for essai in range(10):
        try:
            reponse = rq.get(url, params=params, timeout=180)
        except Exception as e:
            print("Erreur requête :", e, "- pas grave, on va réessayer.")
            time.sleep(2)
            continue
        if reponse.status_code in (200, 206) and reponse.headers.get("content-type", "").startswith("application/json"):
            return reponse.json()
        if 400 <= reponse.status_code < 500 and reponse.status_code != 429:   #Requête refusée : réessayer ne servirait à rien
            raise RuntimeError(f"Hub'eau a refusé la requête ({reponse.status_code}) : {reponse.text[:200]}")
        time.sleep(2 * (essai + 1))   #Serveur saturé : on attend un peu plus à chaque fois
    raise RuntimeError("Hub'eau ne répond pas.")

def telecharger_paquet(paquet, debut="2015-01-01", fin="2026-01-01"):
    data = requete_hubeau({"code_bss": ",".join(paquet), "size": 20000, "format": "json", "date_debut_mesure": debut, "date_fin_mesure": fin})
    if data["count"] <= 20000:   #Tout tient dans une seule page
        return data["data"]
    if len(paquet) > 1:   #Trop de lignes : on coupe le paquet de stations en deux
        return telecharger_paquet(paquet[:len(paquet) // 2], debut, fin) + telecharger_paquet(paquet[len(paquet) // 2:], debut, fin)
    milieu = str(pd.Timestamp(debut) + (pd.Timestamp(fin) - pd.Timestamp(debut)) / 2)[:10]   #Une seule station : on coupe la période en deux
    return telecharger_paquet(paquet, debut, milieu) + telecharger_paquet(paquet, milieu, fin)

def telecharger_chroniques(liste_codes_bss_stations):
    chroniques = []
    for i in range(0, len(liste_codes_bss_stations), 5):   #Paquets de 5 stations : environ 4 000 mesures par station sur 11 ans
        chroniques.extend(telecharger_paquet(liste_codes_bss_stations[i:i+5]))
        print("Stations téléchargées :", min(i + 5, len(liste_codes_bss_stations)), "sur", len(liste_codes_bss_stations))
    #Les deux moitiés d'une période coupée en deux partagent leur jour de coupure : on enlève ces doublons
    return pd.DataFrame(chroniques).drop_duplicates(subset=["code_bss", "date_mesure", "timestamp_mesure"])


morceaux = []
for dep in DEPARTEMENTS:
    lien_chroniques = f"../../data/chroniques_nappes_{dep}.csv"   #Un fichier en cache par département
    if not os.path.exists(lien_chroniques):
        #On ne télécharge que les stations qui peuvent être mesurées au moins 80 % des mois de 2015-2025 (critère des parties 3 et 4) :
        #début des mesures au plus tard en mars 2017 et fin au plus tôt en octobre 2023. Les autres seraient écartées de toute façon.
        utiles = stations.loc[(stations["departement"] == dep) & (stations["date_debut_mesure"] <= "2017-03-31") & (stations["date_fin_mesure"] >= "2023-10-01")]
        telecharger_chroniques(utiles["code_bss"].tolist()).to_csv(lien_chroniques, encoding='utf-8')
        print("Chroniques du département", dep, "téléchargées.")
    morceaux.append(pd.read_csv(lien_chroniques, index_col=0))
chroniques = pd.concat(morceaux, ignore_index=True)
print("Chroniques chargées :", chroniques.shape[0], "mesures.")

On nettoie les données des séries temporelles des nappes phréatiques.

In [ ]:
chroniques['date_mesure'] = pd.to_datetime(chroniques['date_mesure'])
print("Nombre de chroniques sans niveau d'eau : ", chroniques.loc[chroniques["niveau_nappe_eau"].isnull()].shape[0])
#On supprime les chroniques pour lesquelles le niveau d'eau est NULL
chroniques = chroniques.loc[chroniques["niveau_nappe_eau"].notnull()]

## 1.3 Test des données : exemple de série temporelle

Un exemple de série temporelle pour une station (nappe phréatique).

In [ ]:
def fig_exemple_serie(station_cible='00285X0478/PZ10'):
    """Série temporelle brute du niveau d'une station."""
    df_station = chroniques[chroniques['code_bss'] == station_cible]
    df_station = df_station.sort_values(by='date_mesure')

    plt.figure(figsize=(10, 5))
    plt.plot(
        df_station['date_mesure'], 
        df_station['niveau_nappe_eau'], 
        marker='o',         # Ajoute un point sur chaque mesure réelle
        linestyle='-',      # Relie les points par une ligne
        color='#1f77b4', 
        linewidth=2
    )

    plt.title(f"Évolution du niveau de la nappe - Station {station_cible}", fontsize=14)
    plt.xlabel("Date de mesure", fontsize=12)
    plt.ylabel("Niveau de la nappe d'eau (m)", fontsize=12)
    plt.xticks(rotation=45) # Penche les dates pour éviter qu'elles ne se superposent
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

if AFFICHER:
    fig_exemple_serie()

## 1.4 Nettoyage et standardisation des données

Les séries temporelles de Hub'eau donnent des mesures des nappes phréatiques à des dates différentes entre les stations.  
Pour les calculs futurs, on va standardiser ça en faisant une moyenne par station et par mois.

In [ ]:
chroniques['date_mois_annee'] = chroniques['date_mesure'].dt.to_period('M')

chroniques_niveau_mensuel = (chroniques.groupby(["code_bss", "date_mois_annee"])
              .agg({
                  "niveau_nappe_eau":"mean",
                  "profondeur_nappe":"mean"
              })
              .reset_index()
    )
chroniques_niveau_mensuel["date_mesure"] = chroniques_niveau_mensuel["date_mois_annee"].dt.to_timestamp()
chroniques_niveau_mensuel['date_mesure'] = pd.to_datetime(chroniques_niveau_mensuel['date_mesure'])
print("Les données ont été normalisées en moyenne par mois pour chaque station.")

# 2. Usines et stations : premier regard

## 2.1 Répartition des usines et des stations hydrologiques dans les départements étudiés

In [ ]:
def fig_carte_usines_stations():
    """Carte des usines ICPE (rouge) et des stations de mesure des nappes (bleu)."""
    ax = plt.subplots(figsize=(12, 10))[1]
    installations_indust.to_crs(epsg=3857).plot(ax=ax, color="darkred", markersize=8, label="Usines ICPE", zorder=1)
    stations.to_crs(epsg=3857).plot(ax=ax, color="blue", markersize=8, label="Stations hydrologiques", zorder=2)
    ctx.add_basemap(ax, source=ctx.providers.Esri.WorldTopoMap)
    plt.legend(loc="upper left")
    plt.title("Répartition des usines ICPE et des stations hydrologiques dans les départements étudiés")
    plt.show()

if AFFICHER:
    fig_carte_usines_stations()

## 2.2 Nombre d'usines autour de chaque station

In [ ]:
#On veut le nombre d'usines qui sont dans les 5km de chaque station
r = 5 * 1000

stations_rayon = stations.copy()
stations_rayon['geometry'] = stations_rayon.geometry.buffer(r)  #On suppose que la station prend toute la place dans un rayon de 5km autour du point où elle est réellement

usines_ds_rayon = gpd.sjoin(installations_indust, stations_rayon, how="inner", predicate="within")
nb_usine_ds_rayon = usines_ds_rayon.groupby('code_bss').size().reset_index(name='nb_usines_5km')

stations = stations.merge(nb_usine_ds_rayon, on='code_bss', how='left')
stations['nb_usines_5km'] = stations['nb_usines_5km'].fillna(0).astype(int) #On remplace les valeurs nulles par 0

## 2.3 On relie chaque usine à sa station la plus proche

In [ ]:
#On associe chaque usine à la station la plus proche (si elle est à moins de 5km)
usines_station = gpd.sjoin_nearest(installations_indust, stations[['code_bss', 'geometry']], how="left", max_distance=r, distance_col="distance_station")
usines_station = usines_station.drop_duplicates(subset='code_aiot')  #Plusieurs stations peuvent être au même endroit, on n'en garde qu'une
usines_station = usines_station.drop(columns='index_right')
print(usines_station['code_bss'].notnull().sum(), "usines sur ", usines_station.shape[0], "ont une station à moins de 5km.")

## 2.4 On relie chaque nappe aux usines qui sont à moins de 5 km d'une de ses stations

In [ ]:
#Une station peut mesurer plusieurs nappes : on fait une ligne par couple (usine, nappe)
usines_nappes = usines_ds_rayon[['code_aiot', 'nom_ets', 'codes_masse_eau_edl', 'noms_masse_eau_edl']].explode(['codes_masse_eau_edl', 'noms_masse_eau_edl'])
usines_nappes = usines_nappes.rename(columns={'codes_masse_eau_edl': 'code_nappe', 'noms_masse_eau_edl': 'nom_nappe'})
usines_nappes = usines_nappes.dropna(subset=['code_nappe'])  #On enlève les stations sans nappe
usines_nappes = usines_nappes.drop_duplicates(subset=['code_nappe', 'code_aiot'])  #Une usine proche de 2 stations d'une même nappe ne compte qu'une fois

#Pour chaque nappe, la liste des usines à moins de 5km d'une de ses stations
nappes_usines = usines_nappes.groupby(['code_nappe', 'nom_nappe']).agg(nb_usines=('code_aiot', 'size'), usines=('nom_ets', list)).reset_index()
nappes_usines = nappes_usines.sort_values('nb_usines', ascending=False)
print("On a ", nappes_usines.shape[0], "nappes reliées à au moins une usine.")
nappes_usines

## 2.5 Évolution moyenne des nappes selon la pression industrielle

On classe chaque station selon le nombre d'usines à moins de 5 km, puis on suit le niveau moyen de chaque groupe dans le temps.

In [ ]:
#On ajoute à chaque mesure mensuelle la pression industrielle de sa station (nb d'usines dans un rayon de 5km qui exploitent probablement la nappe)
chroniques_pression = chroniques_niveau_mensuel.merge(stations[["code_bss", "nb_usines_5km"]], on="code_bss", how="inner")

conditions = [
    (chroniques_pression['nb_usines_5km'] == 0),
    (chroniques_pression['nb_usines_5km'] > 0) & (chroniques_pression['nb_usines_5km'] <= 5),
    (chroniques_pression['nb_usines_5km'] > 5)
]
choix = ['0 usine (Zone rurale)', '1 à 5 usines (Intermédiaire)', '> 5 usines (Forte pression)']
chroniques_pression['categorie_industrielle'] = np.select(conditions, choix, default='Inconnu')

evolution_lisse = chroniques_pression.groupby(['date_mois_annee', 'categorie_industrielle'])['niveau_nappe_eau'].mean().unstack()

In [ ]:
def fig_evolution_par_pression():
    """Niveau moyen (en m NGF) des stations de chaque groupe de pression industrielle, mois par mois."""
    evolution_lisse.plot(figsize=(12, 6), linewidth=2)
    plt.title("Évolution moyenne des nappes selon la pression industrielle", fontsize=14)
    plt.ylabel("Niveau d'eau moyen (m NGF)")
    plt.xlabel("Date")
    plt.show()

if AFFICHER:
    fig_evolution_par_pression()

On observe que, en moyenne, les nappes qui ont plus de 5 usines à moins de 5 km ont un niveau inférieur d'environ 20 m.  
Nous n'avons pourtant pas établi de causalité : `niveau_nappe_eau` est une altitude (par rapport au niveau de la mer), donc cet écart reflète d'abord l'endroit où se trouvent les stations. 
Les sauts brusques des courbes viennent aussi de ce que l'ensemble des stations mesurées change d'un mois à l'autre.

Les nappes phréatiques sont affectées majoritairement par 3 choses :  
- l'altitude, la topographie et la géologie (partie 3) ;  
- la pluie (partie 4) ;  
- l'industrie (partie 5).

# 3. Conditions géographiques : altitude, roche, profondeur

Le niveau d'une nappe dépend de trois choses : la **géographie** (altitude, roche, profondeur de l'ouvrage), la **pluie** et l'**industrie**.
Avant de chercher l'effet de l'industrie, on regarde si la géographie suffit à expliquer les **variations** des nappes :
si deux nappes de même roche ou de même profondeur bougeaient de la même façon, il faudrait comparer les stations à géographie égale.

On se restreint aux stations bien suivies sur 2015-2025 (au moins 80 % des mois mesurés), les mêmes que dans la partie climat.

In [ ]:
#Niveau mensuel par station sur 2015-2025 : on garde les stations mesurées au moins 80 % des mois
MOIS = pd.date_range("2015-01-01", "2025-12-01", freq="MS")   #Les 132 mois de l'étude, datés du 1er du mois (MS = Month Start)
niveaux_mensuels = chroniques_niveau_mensuel.pivot(index="date_mesure", columns="code_bss", values="niveau_nappe_eau")
niveaux_mensuels = niveaux_mensuels.reindex(MOIS)   #Un mois sans mesure devient une ligne vide (NaN) au lieu de disparaître et les eventuels mois hors période disparaissent

part_mois_mesures = niveaux_mensuels.notna().mean()   
# Moyenne des 1 et 0 selon s'il y a une valeur : donne la proportion des mesures présentes
# C'est une série qui pour chaque code_bss donne une proportion

niveaux_mensuels = niveaux_mensuels.loc[:, part_mois_mesures >= 0.8]
# On garde toutes les lignes des colonnes qui ont au moins 80% de valeurs non Nan


# On comble les trous par interpolation linéaire, seulement entre deux mesures et sur 2 mois d'affilée au plus
# (dans un trou plus long, seuls les 2 premiers mois sont comblés, les suivants restent vides)
# Justifié car le niveau d'eau d'une nappe varie lentement
niveaux_mensuels = niveaux_mensuels.interpolate(limit=2, limit_area="inside")

#Variation de chaque station : amplitude max - min sur 2015-2025 (en mètres, directement lisible ; l'écart-type donne les mêmes conclusions)
stations["amplitude"] = stations["code_bss"].map(niveaux_mensuels.max() - niveaux_mensuels.min())   #NaN pour les stations écartées par le filtre
stations["amplitude"].describe().round(2)

## 3.1 Altitude

Commençons par un état des lieux de l'altitude des différentes stations.

In [ ]:
stations['altitude_station'] = pd.to_numeric(stations['altitude_station'])

def fig_carte_altitude():
    """Carte des stations colorées selon leur altitude."""
    points_carte = stations.to_crs(epsg=3857)   #on transforme le geopandas avec le code EPSG utilisé par les cartes et Google

    fig, ax = plt.subplots(figsize=(12, 10))
    points_carte.plot(ax=ax,
        column='altitude_station', # variable qui donne la couleur du point
        cmap='terrain', # palette de couleurs adaptée à la topographie
        legend=True,                 # Affichage de la barre de gradient
        legend_kwds={
            'label': "Altitude de la station (en mètres)",
            'shrink': 0.7   #Limite la taille de la barre de la légende
        },
        zorder=2
    )
    ctx.add_basemap(ax, source=ctx.providers.Esri.WorldTopoMap, zorder=1)   #Fond de carte
    plt.title("Altitude des stations hydrologiques")
    plt.show()

if AFFICHER:
    fig_carte_altitude()

`niveau_nappe_eau` est une altitude (en m NGF, par rapport au niveau de la mer), pas une hauteur au-dessus du fond de la nappe : on ne peut pas obtenir le volume d'eau dans la nappe, seulement observer son niveau. Une station haut perchée a donc une nappe haute.  
Pour comparer des stations d'altitudes différentes, la suite de l'analyse étudie les **variations** du niveau de chaque station (amplitude, évolution entre deux périodes, tendance) plutôt que le niveau lui-même.  
On regarde d'abord le niveau moyen, puis l'amplitude.

In [ ]:
#Niveau moyen de chaque station (en m NGF, sur toutes ses mesures mensuelles)
#map garde toutes les stations : celles sans chronique ont simplement un niveau moyen vide (NaN), aucune station n'est perdue
stations["niveau_moyen"] = stations["code_bss"].map(chroniques_niveau_mensuel.groupby("code_bss")["niveau_nappe_eau"].mean())

#Dans le Nord, une nappe libre ne descend pas sous le niveau de la mer : un niveau moyen négatif signale une nappe captive rabattue par les pompages
stations["nappe_captive"] = stations["niveau_moyen"] < 0

print(stations["nappe_captive"].sum(), "stations ont un niveau moyen sous le niveau de la mer :")
print(stations.loc[stations["nappe_captive"], ["code_bss", "nom_commune", "profondeur_investigation", "niveau_moyen"]])

In [ ]:
def fig_niveau_altitude():
    """Niveau moyen de la nappe selon l'altitude de la station, nappes captives en rouge."""
    plt.figure(figsize=(8, 6))
    plt.scatter(stations["altitude_station"], stations["niveau_moyen"], c=stations["nappe_captive"].map({True: "darkred", False: "tab:blue"}))   #Les stations sans mesure (NaN) ne sont pas tracées
    plt.plot([0, stations["altitude_station"].max()], [0, stations["altitude_station"].max()], color="grey", linestyle="--", label="Niveau = altitude du sol")
    plt.axhline(0, color="black", linewidth=0.8)
    plt.legend()
    plt.title("Niveau moyen de la nappe selon l'altitude de la station (rouge : sous le niveau de la mer)")
    plt.xlabel("Altitude de la station (m)")
    plt.ylabel("Niveau moyen de la nappe (m NGF)")
    plt.show()

if AFFICHER:
    fig_niveau_altitude()

Le niveau moyen suit l'altitude : la nappe est en général à quelques mètres sous le sol. Les exceptions, sous le niveau de la mer, sont des nappes **captives**
(enfermées sous une couche imperméable) rabattues par des décennies de pompages : le calcaire carbonifère au nord de Lille (vers -60 m NGF) et les sables du Landénien
sous l'argile des Flandres (Nieppe). On les repère avec la colonne `nappe_captive` **sans les écarter** : leur baisse peut justement venir des prélèvements,
industriels compris, ce que l'on cherche à mesurer.

L'altitude fixe donc le niveau moyen. Fixe-t-elle aussi l'ampleur des variations ?

In [ ]:
#Corrélation de rang (Spearman) : ne suppose pas un lien linéaire et résiste aux quelques stations extrêmes ; nan_policy ignore les stations sans amplitude
rho, p = spearmanr(stations["altitude_station"], stations["amplitude"], nan_policy="omit")
print(f"Corrélation de Spearman entre altitude et amplitude : rho = {rho:.2f} (p = {p:.3f})")

def fig_amplitude_altitude():
    """Amplitude 2015-2025 de chaque station selon son altitude."""
    plt.figure(figsize=(8, 6))
    plt.scatter(stations["altitude_station"], stations["amplitude"])
    plt.title("Amplitude de la nappe selon l'altitude de la station")
    plt.xlabel("Altitude de la station (m)")
    plt.ylabel("Amplitude 2015-2025 (max - min, en m)")
    plt.show()

if AFFICHER:
    fig_amplitude_altitude()

**Lecture.** À première vue, l'amplitude monte avec l'altitude : la corrélation de Spearman est positive et significative (**ρ = 0,48**, p = 0,001, sur 45 stations).
Mais le nuage n'a pas la forme d'une tendance : les faibles amplitudes (2 à 2,5 m) sont presque toutes serrées entre 15 et 20 m d'altitude,
et au-delà de 100 m on trouve aussi bien 2 m que 17,6 m. Ce paquet de points bas est un groupe de stations particulier, que la roche permet d'identifier (3.2) :
on refait la corrélation sans lui en 3.3.

## 3.2 Roche

La roche pourrait changer la façon dont la nappe réagit : la craie, épaisse et poreuse, a une forte inertie, un sable superficiel réagit vite.
On classe les stations par famille géologique (`famille_geol`, d'après la masse d'eau) et, pour la partie 3.3, par classe de profondeur de l'ouvrage (`classe_profondeur`).

In [ ]:
#Famille géologique d'après le nom de la masse d'eau (une liste depuis la cellule de chargement : une station peut en croiser plusieurs, on les met bout à bout)
noms_nappes = stations["noms_masse_eau_edl"].str.join(" + ")
#Le calcaire carbonifère passe en premier : une station qui le croise sous la craie mesure en fait le carbonifère (niveau très négatif)
stations["famille_geol"] = np.select(
    [noms_nappes.str.contains("carbonifère", na=False),
     noms_nappes.str.contains("Craie", na=False),
     noms_nappes.str.contains("Sables", na=False),
     noms_nappes.str.contains("Calcaire", na=False),
     noms_nappes.str.contains("Socle", na=False)],
    ["Calcaire carbonifère", "Craie", "Sables du Landénien", "Calcaires de l'Avesnois", "Socle ardennais"],
    default="Non renseignée")

#Profondeur de l'ouvrage (pas celle de l'eau) : un forage profond capte en général une nappe plus inerte
stations["classe_profondeur"] = pd.cut(stations["profondeur_investigation"], bins=[0, 30, 80, np.inf],
                                       labels=["1. superficielle (< 30 m)", "2. intermédiaire (30-80 m)", "3. profonde (> 80 m)"])

#Répartition des stations qui ont des mesures (les autres n'entrent pas dans la suite)
avec_mesures = stations.loc[stations["niveau_moyen"].notna()]
pd.crosstab(avec_mesures["famille_geol"], avec_mesures["classe_profondeur"], margins=True, margins_name="Total")

La craie domine. Les 16 stations « Non renseignée » sont des piézomètres de 5 à 8 m de profondeur de la plaine de la Scarpe (Marchiennes, Vred, Rieulay...),
sans masse d'eau dans Hub'eau : ce sont des nappes superficielles, que l'on garde dans leur propre groupe plutôt que de leur inventer une roche.
Les deux classements se recoupent fortement (le calcaire carbonifère est toujours profond, la craie jamais au-delà de 80 m) et les effectifs par case sont petits :
on comparera par famille **ou** par classe de profondeur, pas les deux à la fois.

In [ ]:
#Seulement les stations suivies sur 2015-2025 (celles qui ont une amplitude)
suivies = stations.dropna(subset=["amplitude"])
amplitude_par_roche = suivies.groupby("famille_geol")["amplitude"].agg(nb_stations="count", mediane="median", minimum="min", maximum="max").round(2)

def fig_amplitude_roche():
    """Boîtes à moustaches de l'amplitude 2015-2025 par famille géologique."""
    suivies.boxplot(column="amplitude", by="famille_geol", grid=False, rot=15, figsize=(10, 6))
    plt.title("Amplitude de la nappe selon la roche")
    plt.suptitle("")   #Sinon pandas ajoute un sous-titre automatique
    plt.xlabel("")
    plt.ylabel("Amplitude 2015-2025 (max - min, en m)")
    plt.show()

if AFFICHER:
    fig_amplitude_roche()
amplitude_par_roche

**Lecture.** Les médianes vont de **2,3 m** (« Non renseignée ») à **3,9 m** (craie), mais l'écart *à l'intérieur* d'une roche est bien plus grand qu'entre roches :
dans la craie, l'amplitude va de **1,7 m à 11 m** (19 stations), dans les calcaires de l'Avesnois de 2,9 à 17,6 m. Les autres familles n'ont que 3 à 5 stations, trop peu pour les départager.
Le seul groupe nettement à part est « Non renseignée » : les **14 piézomètres** de 5 à 8 m de la plaine de la Scarpe, tous sous 3,7 m d'amplitude.
Une nappe superficielle de plaine humide est drainée par les cours d'eau et les fossés dès qu'elle monte, elle reste donc dans une fourchette étroite (hypothèse non vérifiée).

## 3.3 Profondeur

Un forage profond capte en général une nappe plus inerte, un piézomètre de quelques mètres une nappe qui réagit vite.
On trace l'amplitude selon la profondeur de l'ouvrage, en distinguant les piézomètres de la Scarpe (famille « Non renseignée »).

In [ ]:
#Même corrélation avec et sans les piézomètres de la Scarpe, pour la profondeur et pour l'altitude (vue en 3.1)
hors_scarpe = suivies[suivies["famille_geol"] != "Non renseignée"]
lignes_corr = []
for colonne in ["profondeur_investigation", "altitude_station"]:
    rho, p = spearmanr(suivies[colonne], suivies["amplitude"])
    rho_hs, p_hs = spearmanr(hors_scarpe[colonne], hors_scarpe["amplitude"])
    print(f"{colonne} : rho = {rho:.2f} (p = {p:.3f}) sur {len(suivies)} stations, rho = {rho_hs:.2f} (p = {p_hs:.3f}) sans la Scarpe ({len(hors_scarpe)} stations)")
    lignes_corr.append({"variable": colonne,
                        f"rho ({len(suivies)} stations)": round(rho, 2), "p": round(p, 3),
                        f"rho sans la Scarpe ({len(hors_scarpe)} stations)": round(rho_hs, 2), "p sans la Scarpe": round(p_hs, 3)})
correlations_amplitude = pd.DataFrame(lignes_corr).set_index("variable")

amplitude_par_profondeur = suivies.groupby("classe_profondeur", observed=True)["amplitude"].agg(nb_stations="count", mediane="median").round(2)

def fig_amplitude_profondeur():
    """Amplitude selon la profondeur de l'ouvrage, piézomètres de la Scarpe en orange."""
    plt.figure(figsize=(8, 6))
    plt.scatter(suivies["profondeur_investigation"], suivies["amplitude"], c=(suivies["famille_geol"] == "Non renseignée").map({True: "darkorange", False: "tab:blue"}))
    plt.axvline(30, color="grey", linestyle="--")   #Seuils de classe_profondeur
    plt.axvline(80, color="grey", linestyle="--")
    plt.title("Amplitude de la nappe selon la profondeur de l'ouvrage (orange : piézomètres de la Scarpe)")
    plt.xlabel("Profondeur de l'ouvrage (m)")
    plt.ylabel("Amplitude 2015-2025 (max - min, en m)")
    plt.show()

if AFFICHER:
    fig_amplitude_profondeur()
amplitude_par_profondeur

**Lecture.** Sur les 45 stations, l'amplitude augmente avec la profondeur (**ρ = 0,40**, p = 0,007) et la médiane passe de 2,3 m (superficielles) à 3,2 m puis 3,5 m.
Mais ce lien tient entièrement aux piézomètres de la Scarpe (en orange), tous à moins de 8 m et peu variables : sans eux, la corrélation tombe à **ρ = 0,09** (p = 0,62).
Parmi les ouvrages de moins de 30 m, on trouve d'ailleurs à la fois les plus petites amplitudes et l'une des plus grandes (11 m à Lille).
Même chose pour l'altitude vue en 3.1 : sans la Scarpe, ρ = 0,32 (p = 0,08), ce n'est plus significatif au seuil de 5 %.

## Conclusion partielle

La géographie fixe le **niveau moyen** d'une nappe : il suit l'altitude, sauf pour les 4 nappes captives rabattues sous le niveau de la mer.
Elle n'explique pas, en revanche, l'ampleur de ses **variations** : à roche ou profondeur égale, une station peut varier de 2 m comme de 11 m,
et les corrélations avec l'altitude et la profondeur disparaissent dès que l'on met à part un seul groupe homogène, les piézomètres superficiels de la Scarpe, qui varient moins que les autres.

Ces variations viennent d'abord de la météo : les saisons, puis les années sèches et la recharge record de 2023-2024.
L'amplitude mélange cette part climatique et une éventuelle baisse de fond, celle que l'on cherche : la partie suivante retire donc d'abord l'effet de la pluie.

# 4. Partie climat : retirer l'effet de la pluie

Hormis la géographie, il y a aussi le climat qui joue beaucoup sur les variations de niveau des nappes : le niveau monte quand il pleut et diminue quand le sol est sec et a besoin d'eau. Pour juger l'effet de l'industrie toutes choses égales par ailleurs, il faut donc aussi retirer les effets de la météo. 

Le résidu qui restera sera la partie que la pluie n'explique pas. 
Ce retrait reste une estimation car une partie de l'effet humain peut passer pour du climat.

## 4.1 Pluie efficace SAFRAN

Météo-France publie en open data la réanalyse SAFRAN : une valeur par jour sur une grille de 8 km.
On garde la pluie efficace PE (pluie moins évaporation, en mm) de la maille la plus proche de chaque station.
On remonte à 2012 car une nappe a de la mémoire : elle réagit aux pluies du passé.

In [ ]:
URL_SAFRAN = "https://meteofrance.s3.sbg.io.cloud.ovh.net/data/REF_CC/SIM/QUOT_SIM2_{}.csv.gz"   #Les {} seront remplacées par l'année
lien_pluie = "../../data/pluie_efficace_stations.csv.gz"   #Cache : une ligne par station et par mois

#SAFRAN est en Lambert II étendu (EPSG:27572), avec des coordonnées en hectomètres (x 100 pour repasser en mètres)
stations_lambert2 = (stations
                     .drop_duplicates(subset="code_bss")
                     .set_index("code_bss")
                     .to_crs(epsg=27572)
                     [["geometry"]])   #Doubles crochets : on garde un GeoDataFrame (et pas une simple série de points), sjoin_nearest en a besoin

#Téléchargement, seulement la première fois (environ 2 Go) : ensuite on relit le fichier local
if not os.path.exists(lien_pluie):
    # Code légèrement sous optimal pour une question de facilité de lecture du code et de compréhension : on telecharge 2 fois 2012 mais c'est minime
    # On prend le fichier des pluies et on extrait la grille, ensuite on fait l'appariement avec les stations de la partie précédente et 
    # enfin on garde seulement les mailles utiles. Après on fait des jointures entre les mailles utiles calculées qui nous intéresse et les pluies


    #Chaque maille apparaît une fois par jour dans le fichier, on ne garde que la liste des mailles
    grille = (pd.read_csv(URL_SAFRAN.format(2012), sep=";", usecols=["LAMBX", "LAMBY"])
                .rename(columns={"LAMBX": "X", "LAMBY": "Y"})
                .drop_duplicates())
    
    #Type nécessaire pour faire le sjoin_nearest et passage des hm en m
    grille = gpd.GeoDataFrame(grille, geometry=gpd.points_from_xy(grille["X"] * 100, grille["Y"] * 100), crs="EPSG:27572")
    
    appariement = gpd.sjoin_nearest(stations_lambert2, grille, distance_col="distance")

    appariement = appariement.reset_index().drop_duplicates(subset="code_bss")   #En cas d'égalité de distance, on garde une seule maille par station
    appariement = appariement[["code_bss", "X", "Y", "distance"]]     # On ne conserve que les colonnes qui nous intéressent
    
    #Les mailles utiles : plusieurs stations partagent souvent la même maille, on ne garde chaque maille qu'une fois,
    mailles_utiles = appariement[["X", "Y"]].drop_duplicates()


    def pluie_mensuelle(annee):
        """Pluie efficace mensuelle des mailles utiles pour une année (un fichier par an pour toute la France, environ 140 Mo)."""
        safran_annee = pd.read_csv(URL_SAFRAN.format(annee), sep=";", usecols=["LAMBX", "LAMBY", "DATE", "PE"]).rename(columns={"LAMBX": "X", "LAMBY": "Y"})

        #On ne garde que les lignes SAFRAN des mailles utiles pour réduire les coûts de calcul, c'est l'intérêt de mailles_utiles
        safran_annee = safran_annee.merge(mailles_utiles, how="inner", on=["X", "Y"])

        #Cumul mensuel de la pluie efficace par maille
        #SAFRAN écrit les dates sous la forme 20120101 ; on supprime le jour en enlevant les 2 derniers chiffres
        safran_annee["mois"] = safran_annee["DATE"]//100
        return safran_annee.groupby(["X", "Y", "mois"])["PE"].sum().reset_index()

    # On part de 2012 car le modèle utilise jusqu'à 36 mois de pluie avant 2015. On télécharge 4 années à la fois pour aller plus vite
    # L'étape lente est le téléchargement : le programme attend surtout le réseau, il ne calcule presque pas.
    # Les threads en profitent : pendant qu'un téléchargement attend, un autre avance.
    from concurrent.futures import ThreadPoolExecutor
    # Pool de 4 threads : au plus 4 années en même temps, assez pour gagner du temps sans saturer la mémoire (~140 Mo par fichier).
    # Le with ferme le pool proprement à la fin, après avoir attendu que tous les threads aient terminé.
    with ThreadPoolExecutor(max_workers=4) as executeur:
        # Même principe que map() : pluie_mensuelle est appelée sur chaque année de 2012 à 2025 (2026 exclu), chaque appel confié à un ouvrier libre.
        # Les résultats reviennent dans l'ordre des années ; list() attend la fin de tout et donne 14 DataFrames, recollés plus bas par pd.concat.
        morceaux = list(executeur.map(pluie_mensuelle, range(2012, 2026)))


    # On stocke tout dans un même fichier qui contient les colonnes : X, Y, date/mois, pluie efficace, code BSS des stations et distance (entre le centre de la grille et la station)
    # Il y a plein de lignes pour une même station ou un même mois
    # si on veut les appariements, on drop_duplicates sur la colonne code_bss en ne gardant que les colonnes X, Y, distance
    # si on veut la pluie efficace mensuelle, on fait .pivot en prenant les mois pour index, les stations pour colonnes et la pluie efficace comme valeur

    cache_csv = appariement.merge(pd.concat(morceaux), on=["X", "Y"])
    cache_csv["mois"] = pd.to_datetime(cache_csv["mois"].astype(str), format="%Y%m") #sans jour donné dans le str, pandas met le 1er du mois, ce qu'il nous faut pour la suite
    # Au prochain lancement, on relit ce petit fichier au lieu de retélécharger 2 Go
    cache_csv[["code_bss", "X", "Y", "distance", "mois", "PE"]].to_csv(lien_pluie, index=False, compression="gzip")

cache_csv = pd.read_csv(lien_pluie, parse_dates=["mois"])

pluie_eff_mensuelle = cache_csv.pivot(index="mois", columns="code_bss", values="PE")   #Une ligne par mois, une colonne par station
appariement = cache_csv.drop_duplicates(subset="code_bss").set_index("code_bss")[["X", "Y", "distance"]]   #Une ligne par station : sa maille

## 4.2 Modèle climat → nappe

Pour chaque station, on explique le niveau mensuel par une régression linéaire :

$$\text{niveau}_t = a + b \, PE_{k \text{ derniers mois}} + c \, PE_{3 \text{ derniers mois}} + e \, PE_{6 \text{ derniers mois}} + \text{saison (sinus, cosinus)} + d \, \text{temps}$$


On ajuste par station pour bien retirer ce qui est propre à la nappe, c'est-à-dire ses caractéristiques propres qui rentrent dans la constante, ainsi que sa localisation qui détermine la quantité de pluie qu'elle reçoit, grandeur que l'on essaie de retirer pour mieux observer ses variations. 

Comme vu à la partie précédente, les nappes ne sont pas toutes identiques en termes de caractéristiques : on essaie plusieurs valeurs de « mémoire » pour modéliser les différents types de roches et les différentes profondeurs (et bien sûr les différentes tailles de nappes que la partie précédente n'a pas pu mettre en lumière). On essaie k = 12, 24 et 36 mois et on garde le meilleur.

Le terme $d \times t$ ($t$ = temps en années) capte la tendance temporelle mais non climatique. Sans lui, la régression attribuerait à la pluie
toute la part de la baisse qui évolue en même temps qu'elle (années sèches puis recharge record de 2023-2024) et sous-estimerait la tendance.

On estime tout ensemble, puis on garde seulement $d \times t$ de la simulation : la partie_climat = constante + pluie + saison est retirée.

$a$ porte la géographie de la station : l'enlever du résidu permet de comparer les stations.

Le R² mesure la qualité du modèle complet (climat + tendance) : on garde le plus grand comme critère pour le choix de k.

La tendance est mesurée par la médiane des pentes entre les paires d'un même mois (janvier 2015 avec janvier 2020...).
On a pris ce parti car nous estimons que comparer deux mois différents mélangerait la tendance et le cycle saisonnier.

In [ ]:
# Régression linéaire pour trouver, station par station, la part du niveau expliquée par la pluie
# On maximise le R² pour expliquer le plus possible les variations par le climat 
# de telle sorte que le résidu est expliqué presque entièrement par le reste : l'action humaine ou d'autres causes. 
def modele_climat(st):
    """Régression du niveau sur la pluie efficace cumulée et le temps ; renvoie (R², k, niveau simulé par le climat seul)."""
    pe, y = pluie_eff_mensuelle[st], niveaux_mensuels[st] 

    meilleur = (-np.inf, None, None) #initialisation du triplet recherché

    for k in [12, 24, 36]:   #k = 6 ferait doublon avec pe_6
        X = pd.DataFrame({"pe_k": pe.rolling(k).sum(), "pe_3": pe.rolling(3).sum(), "pe_6": pe.rolling(6).sum()}).reindex(MOIS)
        # on définit un dataframe pour notre régression qu'on initialise avec la pluie efficace des derniers mois en testant plusieurs combinaisons
        # grâce à des sommes glissantes et avec une reindexation après

        X["sin"], X["cos"], X["constante"] = np.sin(2 * np.pi * MOIS.month / 12), np.cos(2 * np.pi * MOIS.month / 12), 1.0 
        # on définit les variables pour la régression : sin(2*pi*n°mois) et cos(2*pi*n°mois) pour capter les non linéarités dues aux saisons 
        # et la constante pour le biais

        X["tendance"] = (MOIS - MOIS[0]).days / 365.25
        # temps en années depuis janvier 2015 : la tendance non climatique est estimée EN MÊME TEMPS que l'effet de la pluie,
        # sinon la régression attribue à la pluie la part de la baisse qui évolue avec elle (années sèches puis recharge 2023-2024)

        non_vide = y.notna()

        beta = np.linalg.lstsq(X[non_vide], y[non_vide], rcond=None)[0]
        complet = X @ beta
        # produit "matriciel" entre le df des variables explicatives (X) et l'estimateur des MCO (beta)
        # on a donc chaque colonne multiplié par un coefficient et on somme les colonnes entre elles pour obtenir une colonne qui est la prédiction du modèle complet

        partie_climat = complet - beta[-1] * X["tendance"]
        # niveau simulé sans la tendance (dernière colonne de X) : climat + constante (altitude), la tendance reste donc dans le résidu observé - simulé

        r2 = 1 - ((y - complet) ** 2).sum() / ((y - y.mean()) ** 2).sum() # calcul du R² du modèle complet, 
        # plus il est proche de 1 plus on explique bien les variations avec nos variables explicatives
        # on regarde s'il est meilleur que celui d'avant et si oui on actualise, le but est de trouver les parametres qui maximise R²
        if r2 > meilleur[0]:
            meilleur = (r2, k, partie_climat)
    return meilleur

In [ ]:
def pente_sen(serie):
    """Pente de Sen saisonnière en cm/an : médiane des pentes entre les paires (i < j) du même mois calendaire.
    Deux mois différents ne sont jamais comparés : le cycle saisonnier ne peut pas fausser la pente."""
    s = serie.dropna()
    x = list((s.index - s.index[0]).days / 365.25)   #Temps en années depuis la première mesure
    y = list(s.values)
    mois = list(s.index.month)

    pentes = []
    n = len(x)
    for i in range(n):
        for j in range(i + 1, n):
            if mois[j] == mois[i]:   #Même mois d'années différentes, donc x[j] > x[i] : pas de division par zéro
                pentes.append((y[j] - y[i]) / (x[j] - x[i]))

    pentes.sort()
    m = len(pentes)
    if m % 2 == 1:
        mediane = pentes[m // 2]
    else:
        mediane = (pentes[m // 2 - 1] + pentes[m // 2]) / 2
    return mediane * 100   #m/an -> cm/an

In [ ]:
lignes, partie_climat = {}, {}   # deux dicts indexés par station (clé = code_bss), remplis dans la même boucle donc dans le même ordre
                                 # lignes[station] = dict des indicateurs de la station -> deviendra une LIGNE de tendances
                                 # partie_climat[station] = Series mensuelle simulée sans la tendance -> deviendra une COLONNE de partie_climat
for station in niveaux_mensuels.columns:
    r2, k, partie_climat[station] = modele_climat(station)
    lignes[station] = {"r2": r2, "k_mois": k}   # il est plus rapide de remplir un dict puis de construire le df une seule fois que d'ajouter des lignes à un df

tendances = pd.DataFrame.from_dict(lignes, orient="index").rename_axis("code_bss")   # orient="index" : les clés deviennent l'index (une ligne par station) ; rename_axis nomme cet index
partie_climat = pd.DataFrame(partie_climat)   # orientation par défaut : les clés deviennent les colonnes, les séries sont alignées sur les dates
#Le retrait du climat (résidu et pentes corrigées) se fait en 4.5, une fois le modèle contrôlé

## 4.3 Contrôle : la mémoire k selon la profondeur et la roche

Si le modèle est physiquement cohérent, la mémoire k retenue doit être plus longue pour les nappes profondes et inertes (craie, captives)
que pour les nappes superficielles, qui réagissent vite à la pluie.

In [ ]:
#Part des stations qui ont retenu chaque k, par classe de profondeur
controle = tendances.join(stations.drop_duplicates(subset="code_bss").set_index("code_bss")["classe_profondeur"])

def fig_memoire_profondeur():
    """Part des stations ayant retenu chaque mémoire k, par classe de profondeur."""
    pd.crosstab(controle["classe_profondeur"], controle["k_mois"], normalize="index").plot.bar(stacked=True, rot=0, figsize=(8, 5), xlabel="", ylabel="Part des stations", title="Mémoire k retenue selon la profondeur de l'ouvrage")
    plt.legend(title="k (mois)", bbox_to_anchor=(1, 1))   #Légende à droite pour ne pas cacher les barres
    plt.show()

if AFFICHER:
    fig_memoire_profondeur()

**Lecture.** Plus l'ouvrage est profond, plus la mémoire retenue est longue. La part des stations à k = 12 mois passe de 56 % (superficielles) à 18 % (intermédiaires) puis 14 % (profondes).
C'est un contrôle de cohérence, pas une preuve, car k ne prend que 3 valeurs et les R² de deux k voisins sont souvent proches.

## 4.4 Contrôle : le cycle saisonnier vient-il du climat ou des pompages ?

`partie_climat` contient le terme saison (sinus, cosinus), retiré du niveau en 4.5. C'est légitime seulement si le cycle annuel des nappes vient surtout du climat, et pas des pompages (plus forts en été).

In [ ]:
#Chaque année, remontée d'hiver (octobre -> avril) et baisse d'été (avril -> octobre), face à la pluie efficace de la même saison
cycles = []
for station in niveaux_mensuels.columns:
    niveau, pe = niveaux_mensuels[station], pluie_eff_mensuelle[station]
    for an in range(2016, 2026):
        cycles.append({"code_bss": station,
                       "montee_hiver": niveau[pd.Timestamp(an, 4, 1)] - niveau[pd.Timestamp(an - 1, 10, 1)], "pe_hiver": pe[f"{an - 1}-10":f"{an}-03"].sum(),
                       "baisse_ete": niveau[pd.Timestamp(an, 10, 1)] - niveau[pd.Timestamp(an, 4, 1)], "pe_ete": pe[f"{an}-04":f"{an}-09"].sum()})
cycles = pd.DataFrame(cycles)

centre = cycles.groupby("code_bss").transform(lambda s: s - s.mean())   #Chaque station centrée : on ne compare que ses années entre elles

def fig_cycle_saisonnier():
    """Remontée d'hiver et baisse d'été face à la pluie efficace de la saison (écarts à la moyenne de chaque station)."""
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    axes[0].scatter(centre["pe_hiver"], centre["montee_hiver"], s=10, alpha=0.4) # s = taille des points, alpha la transparence
    axes[1].scatter(centre["pe_ete"], centre["baisse_ete"], s=10, alpha=0.4)

    axes[0].set(title="Remontée d'hiver (octobre -> avril)", xlabel="Pluie efficace de la saison, écart à la moyenne de la station (mm)", ylabel="Variation du niveau, écart à la moyenne de la station (m)")
    axes[1].set(title="Baisse d'été (avril -> octobre)", xlabel="Pluie efficace de la saison, écart à la moyenne de la station (mm)", ylabel="Variation du niveau, écart à la moyenne de la station (m)")

    axes[0].axhline(0, color="dimgray", linewidth=0.8)  # ajout des axes en gris
    axes[1].axhline(0, color="dimgray", linewidth=0.8)
    axes[0].axvline(0, color="dimgray", linewidth=0.8)  
    axes[1].axvline(0, color="dimgray", linewidth=0.8)

    plt.tight_layout()
    plt.show()

if AFFICHER:
    fig_cycle_saisonnier()

**Lecture.** Dans les deux graphiques, les points montent de gauche à droite : un hiver plus pluvieux que d'habitude fait davantage remonter la nappe, un été plus pluvieux la fait moins baisser.
Le cycle change donc d'une année à l'autre avec la pluie, alors qu'un pompage régulier donnerait à peu près le même chaque année : il vient surtout du climat, et le retirer en 4.5 est justifié.
Réserve : les étés secs sont aussi ceux où l'on pompe le plus, ce qui peut accentuer la baisse d'été.

## 4.5 Retrait de la partie climatique

On retire à chaque station sa partie climatique (constante + pluie + saison) :
le résidu = observé − partie_climat garde la tendance $d \times t$ et tout ce que le climat n'explique pas.
On mesure ensuite la tendance avant (pente brute) et après (pente corrigée) ce retrait, par la pente de Sen saisonnière.

In [ ]:
residu = niveaux_mensuels - partie_climat   #Mêmes mois en lignes, mêmes stations en colonnes
tendances["pente_brute_cm_an"] = niveaux_mensuels.apply(pente_sen)   #apply : pente_sen sur chaque colonne, donc chaque station
tendances["pente_corrigee_cm_an"] = residu.apply(pente_sen)
#Nappes captives (niveau moyen sous la mer) : le modèle suppose une recharge par la pluie locale, leur pente corrigée est à lire avec prudence
tendances["nappe_captive"] = niveaux_mensuels.mean() < 0
tendances.sort_values("pente_corrigee_cm_an").round(2)

In [ ]:
#Résumé des tendances avant et après retrait du climat
resume_tendances = pd.DataFrame({
    "pente brute (cm/an)": [tendances["pente_brute_cm_an"].median(), (tendances["pente_brute_cm_an"] < 0).sum(), (tendances["pente_brute_cm_an"] > 0).sum()],
    "pente corrigée (cm/an)": [tendances["pente_corrigee_cm_an"].median(), (tendances["pente_corrigee_cm_an"] < 0).sum(), (tendances["pente_corrigee_cm_an"] > 0).sum()],
}, index=["médiane", "stations en baisse", "stations en hausse"]).round(2)
print(resume_tendances)
print("Stations dont la pente corrigée est inférieure à la pente brute :", (tendances["pente_corrigee_cm_an"] < tendances["pente_brute_cm_an"]).sum(), "sur", len(tendances))

def fig_pentes_brute_corrigee():
    """Tendance de chaque station avant (x) et après (y) retrait du climat ; nappes captives en rouge."""
    plt.figure(figsize=(8, 7))
    couleurs = tendances["nappe_captive"].map({True: "darkred", False: "tab:blue"})
    plt.scatter(tendances["pente_brute_cm_an"], tendances["pente_corrigee_cm_an"], c=couleurs)
    bornes = [tendances[["pente_brute_cm_an", "pente_corrigee_cm_an"]].min().min(), tendances[["pente_brute_cm_an", "pente_corrigee_cm_an"]].max().max()]
    plt.plot(bornes, bornes, color="grey", linestyle="--", label="Pente corrigée = pente brute")
    plt.axhline(0, color="black", linewidth=0.8)
    plt.axvline(0, color="black", linewidth=0.8)
    plt.legend()
    plt.title("Tendance de chaque station avant et après retrait du climat (rouge : nappes captives)")
    plt.xlabel("Pente brute (cm/an)")
    plt.ylabel("Pente corrigée du climat (cm/an)")
    plt.show()

if AFFICHER:
    fig_pentes_brute_corrigee()

**Lecture.** Toutes les stations sont sous la diagonale : pour chacune, la pente corrigée est plus basse que la pente brute.
Sur la période, la pluie a donc poussé les nappes vers le haut (années humides en fin de période, dont la recharge record de 2023-2024), ce qui masque une partie de la baisse dans le niveau observé.
Avant correction, les stations se partagent entre baisses (26) et hausses (19), avec une médiane proche de zéro (−0,6 cm/an).
Après correction, 41 stations sur 45 sont en baisse, avec une médiane de −4,6 cm/an : à pluie égale, la plupart des nappes baissent légèrement.
Les trois plus fortes baisses corrigées sont des nappes captives (Bondues, Nieppe, Quesnoy-sur-Deûle), pour lesquelles le modèle est à lire avec prudence.
Ce résidu mesure ce que la pluie n'explique pas, sans dire d'où il vient : il n'est pas, à ce stade, relié à la pression industrielle.

**Limite**

Le modèle sous-estime probablement l'effet humain (limite principale).
Les prélèvements ne sont pas répartis au hasard dans le temps : ils sont plus forts en été (irrigation, refroidissement, consommation)
et augmentent les années sèches. Or le modèle a justement deux termes qui varient de la même façon : la saison (sinus, cosinus) et la pluie efficace (faible les années sèches).
Tout ce que les prélèvements ont de saisonnier ou de lié à la sécheresse est donc compté comme « climat » et sort du résidu.

# 5. Effet de l'industrie sur les nappes phréatiques

Objectif : déterminer si les nappes situées près des installations industrielles ont évolué plus défavorablement que les autres entre 2015 et 2025, **une fois l'effet de la pluie retiré** (partie 4).

Démarche :
1. Mesurer l'évolution de chaque nappe, climat retiré : la tendance corrigée de la partie 4.
2. Mesurer la pression industrielle autour de chaque station (nombre d'usines).
3. Comparer les deux, puis chercher ce qui distingue les stations (géologie, profondeur).
4. Vérifier avec les volumes d'eau réellement pompés (BNPE).

## 5.1 Mesure de l'évolution des nappes, climat retiré

Pour chaque station, l'évolution est la **tendance corrigée du climat** calculée en partie 4 : la pente de Sen saisonnière du résidu (niveau observé moins la partie expliquée par la pluie et les saisons), en cm/an.

Cet indicateur a deux avantages sur une simple comparaison entre le début et la fin de la période : il utilise toute la série, donc ne dépend pas du choix des périodes, et la pluie étant retirée, une année exceptionnellement humide comme 2024 ne le fausse plus.
On étudie donc les stations bien suivies de la partie 4 (au moins 80 % des mois mesurés). Pour comparer, on garde aussi la pente brute, calculée de la même façon avant le retrait du climat.

In [ ]:
# On crée le tableau de travail : les stations de la partie 4 avec leur tendance, leur profondeur et leur géologie
st = stations.merge(tendances[["pente_brute_cm_an", "pente_corrigee_cm_an"]].reset_index(), on="code_bss")
st["tendance_cm_an"] = st["pente_corrigee_cm_an"]   #Indicateur d'évolution de toute la partie : la tendance une fois la pluie retirée

st["profondeur_investigation"] = pd.to_numeric(st["profondeur_investigation"], errors="coerce")
st["classe_profondeur"] = pd.cut(st["profondeur_investigation"], bins=[-np.inf, 30, 80, np.inf], labels=["Superficielle (<30m)", "Intermédiaire (30-80m)", "Profonde (>80m)"])

noms = st["noms_masse_eau_edl"].astype(str)
st["famille_geologique"] = np.select(
    [noms.str.contains("Craie", case=False), noms.str.contains("Sable|Landénien", case=False), noms.str.contains("Calcaire", case=False)],
    ["Craie", "Sable", "Calcaire"],
    default="Autre"
)

print("Stations étudiées :", len(st))
print(st["tendance_cm_an"].describe().round(2))

In [ ]:
#Trajectoire annuelle de chaque station sans la pluie : moyenne annuelle du résidu de la partie 4 (années d'au moins 10 mois mesurés),
#en écart à sa moyenne 2015-2017 (on travaille sur des moyennes annuelles pour ne jamais comparer un été à un hiver)
mois_mesures = residu.notna().groupby(residu.index.year).sum()
residu_annuel = residu.groupby(residu.index.year).mean().where(mois_mesures >= 10)   #Une ligne par année, une colonne par station
ecart_annuel = residu_annuel - residu_annuel.loc[2015:2017].mean()

In [ ]:
def fig_distribution_evolutions():
    """Histogramme et classement des tendances corrigées du climat de chaque piézomètre."""
    med = st["tendance_cm_an"].median()
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].hist(st["tendance_cm_an"], bins=30, color="steelblue", edgecolor="white")
    axes[0].axvline(0, color="black", linewidth=1)
    axes[0].axvline(med, color="red", linestyle="--", label=f"Médiane : {med:+.1f} cm/an")
    axes[0].set_xlabel("Tendance corrigée du climat (cm/an)")
    axes[0].set_ylabel("Nombre de piézomètres")
    axes[0].set_title(f"Distribution des tendances ({len(st)} piézomètres)")
    axes[0].legend()

    tri = st["tendance_cm_an"].sort_values().reset_index(drop=True)
    axes[1].bar(tri.index, tri, width=1, color=np.where(tri < 0, "tab:red", "tab:blue"))
    axes[1].axhline(0, color="black", linewidth=1)
    axes[1].axhline(med, color="grey", linestyle="--", label="Médiane")
    axes[1].set_xticks([])
    axes[1].set_xlabel("Piézomètres (triés)")
    axes[1].set_ylabel("Tendance corrigée (cm/an)")
    axes[1].set_title(f"{(tri > 0).sum()} en hausse / {(tri < 0).sum()} en baisse")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

if AFFICHER:
    fig_distribution_evolutions()

**Interprétation.** Une fois la pluie retirée, 41 des 45 stations sont en baisse, avec une médiane de −4,6 cm/an : à pluie égale, la nappe typique du Nord baisse légèrement. La moitié des stations ont une tendance comprise entre −11,8 et −1,9 cm/an, et les 4 stations en hausse le sont de moins de 4 cm/an. Dans les niveaux bruts, ce mouvement est masqué par les années humides de la fin de période (voir partie 4).

Les plus fortes baisses sont Bondues (−57 cm/an), Nieppe (−36 cm/an) et Quesnoy-sur-Deûle (−24 cm/an) : trois nappes captives, pour lesquelles le modèle, qui suppose une recharge par la pluie locale, est à lire avec prudence.

In [ ]:
def fig_plus_fortes_evolutions():
    """Trajectoire annuelle, climat retiré, des 5 plus fortes baisses et des 5 tendances les plus hautes."""
    tri_st = st.sort_values("tendance_cm_an")
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
    for ax, codes, titre in [(axes[0], tri_st["code_bss"].head(5), "Plus fortes baisses"), (axes[1], tri_st["code_bss"].tail(5), "Tendances les plus hautes")]:
        for code in codes:
            tendance = st.loc[st["code_bss"] == code, "tendance_cm_an"].iloc[0]
            ax.plot(ecart_annuel.index, ecart_annuel[code], marker="o", label=f"{code} ({tendance:+.1f} cm/an)")
        ax.axvspan(2015, 2017, color="grey", alpha=0.15)   #Période de référence
        ax.axhline(0, color="black", linewidth=0.8)
        ax.set_title(titre)
        ax.set_xlabel("Année")
        ax.legend(fontsize=8)
    axes[0].set_ylabel("Écart au niveau 2015-2017, pluie retirée (m)")
    plt.tight_layout()
    plt.show()

if AFFICHER:
    fig_plus_fortes_evolutions()

> **À compléter après exécution.** Décrire les trajectoires : les baisses sont-elles régulières ou concentrées sur quelques années ? Bondues se distingue-t-elle toujours des autres ? Une fois la pluie retirée, le mouvement commun lié au climat (baisse de 2017 à 2019-2020, remontée en 2024) ne devrait plus apparaître : le vérifier.

## 5.2 Premier indicateur de pression : le nombre d'usines

On ne garde que l'industrie manufacturière (codes NAF 10 à 33) et l'énergie (code 35), qui sont les activités les plus consommatrices d'eau. Pour chaque station, on compte les usines situées à moins de 2, 5 et 10 km, afin de vérifier que le résultat ne dépend pas du rayon choisi.

Les stations sont ensuite réparties en trois groupes : 0 usine, 1 à 5 usines, plus de 5 usines (dans un rayon de 5 km).

In [ ]:
# On ne garde que l'industrie manufacturière et l'énergie
code_naf = pd.to_numeric(installations_indust["code_naf"].astype(str).str[:2], errors="coerce")
usines = installations_indust[code_naf.between(10, 33) | (code_naf == 35)]
print("Usines manufacturières et énergétiques retenues :", len(usines))

#On compte les usines à proximité de chaque station avec différents rayons
for rayon_km in [2, 5, 10]:
    st[f"usines_{rayon_km}km"] = st.geometry.apply(lambda g: (usines.distance(g) <= rayon_km * 1000).sum())

#On sépare les stations en fonction de la pression industrielle (=nb d'usines aux alentours) qu'elles subissent
st["pression"] = pd.cut(st["usines_5km"], bins=[-1, 0, 5, 10000], labels=["0 usine", "1 à 5 usines", "plus de 5 usines"])
print(st["pression"].value_counts())

In [ ]:
evolution_par_pression = st.groupby("pression", observed=False)["tendance_cm_an"].describe().round(2)
print(evolution_par_pression)

def fig_evolution_par_pression_boxplot():
    """Boîtes à moustaches de la tendance corrigée selon le nombre d'usines à moins de 5 km."""
    st.boxplot(column="tendance_cm_an", by="pression", figsize=(10, 6))
    plt.suptitle("")
    plt.title("Tendance des nappes, climat retiré, selon la pression industrielle (rayon 5 km)")
    plt.xlabel("Nombre d'usines à moins de 5 km")
    plt.ylabel("Tendance corrigée du climat (cm/an)")
    plt.show()

if AFFICHER:
    fig_evolution_par_pression_boxplot()

> **À compléter après exécution.** Comparer les médianes des trois groupes (tableau imprimé ci-dessus) : le groupe « plus de 5 usines » baisse-t-il plus que les autres ? Commenter la médiane plutôt que la moyenne, que quelques stations extrêmes comme Bondues tirent vers le bas. Vérifier l'effectif de chaque groupe : un groupe de quelques stations est à interpréter avec prudence.

In [ ]:
for rayon_km in [2, 5, 10]:
    print(f"Spearman tendance corrigée / usines à {rayon_km} km :", round(st["tendance_cm_an"].corr(st[f"usines_{rayon_km}km"], method="spearman"), 2))

On utilise la corrélation de Spearman : elle mesure si « plus d'usines » va avec « plus de baisse ». Une corrélation **négative** indiquerait que les nappes entourées de plus d'usines baissent davantage.

> **À compléter après exécution.** Donner les trois valeurs (2, 5 et 10 km), leur signe, et dire si le résultat change avec le rayon.

In [ ]:
def fig_carte_evolutions():
    """Carte des stations colorées selon leur tendance corrigée du climat, avec les usines en noir."""
    fig, ax = plt.subplots(figsize=(12, 10))
    usines.to_crs(epsg=3857).plot(ax=ax, color="black", markersize=5, alpha=0.4, label="Usines", zorder=2)
    st.to_crs(epsg=3857).plot(ax=ax, column="tendance_cm_an", cmap="RdBu", norm=TwoSlopeNorm(vmin=-10, vcenter=0, vmax=10), markersize=80, edgecolor="black", legend=True, legend_kwds={"label": "Tendance corrigée du climat (cm/an)"}, zorder=3)
    ctx.add_basemap(ax, source=ctx.providers.Esri.WorldTopoMap, zorder=1)
    ax.set_axis_off()
    plt.legend(loc="upper left")
    plt.title("Tendance des nappes, climat retiré, et localisation des usines")
    plt.show()

if AFFICHER:
    fig_carte_evolutions()

La couleur sature à ±10 cm/an pour que les petites tendances restent visibles.

> **À compléter après exécution.** Les stations en baisse (rouge) se concentrent-elles autour des pôles industriels, en particulier dans la métropole lilloise ?

## 5.3 Ce qui distingue les stations en baisse

On regarde les caractéristiques des stations extrêmes, puis l'effet de la géologie et de la profondeur, pour voir si elles expliquent mieux les différences entre stations que la pression industrielle.

In [ ]:
colonnes = ["code_bss", "nom_commune", "tendance_cm_an", "usines_5km", "classe_profondeur", "famille_geologique"]
stations_extremes = pd.concat([st.sort_values("tendance_cm_an")[colonnes].head(8), st.sort_values("tendance_cm_an")[colonnes].tail(5)])
print("Plus fortes baisses")
print(st.sort_values("tendance_cm_an")[colonnes].head(8).to_string(index=False))
print("\nTendances les plus hautes")
print(st.sort_values("tendance_cm_an")[colonnes].tail(5).to_string(index=False))

> **À compléter après exécution.** Les plus fortes baisses sont-elles toutes entourées d'usines, ou certaines n'en ont-elles aucune ? Bondues et les stations de Lille ont un nombre d'usines comparable : comparer leurs tendances, leur profondeur et leur géologie.

In [ ]:
print(pd.crosstab(st["famille_geologique"], st["classe_profondeur"], margins=True))
print(st.groupby("famille_geologique")["tendance_cm_an"].describe().round(2))
print(st.groupby("classe_profondeur", observed=False)["tendance_cm_an"].describe().round(2))

def fig_evolution_geologie_profondeur():
    """Boîtes à moustaches de la tendance corrigée selon la géologie et selon la profondeur."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
    st.boxplot(column="tendance_cm_an", by="famille_geologique", ax=axes[0])
    st.boxplot(column="tendance_cm_an", by="classe_profondeur", ax=axes[1])
    axes[0].set_title("Selon la géologie")
    axes[1].set_title("Selon la profondeur")
    axes[0].set_xlabel("")
    axes[1].set_xlabel("")
    axes[0].set_ylabel("Tendance corrigée du climat (cm/an)")
    plt.suptitle("Tendance des nappes, climat retiré, selon leurs caractéristiques")
    plt.tight_layout()
    plt.show()

if AFFICHER:
    fig_evolution_geologie_profondeur()

> **À compléter après exécution.** Comparer l'écart entre les médianes selon la géologie et selon la profondeur à l'écart entre les médianes selon la pression industrielle (5.2) : lequel est le plus grand ? Vérifier dans le tableau croisé si les stations profondes sont aussi les calcaires, auquel cas on ne peut pas séparer les deux effets. Rappel : la craie de la partie 3 se recharge vite, les nappes captives profondes lentement.

## 5.4 Deuxième indicateur de pression : les volumes pompés

Le nombre d'usines est un indicateur imparfait : une petite usine et une grande usine comptent pareil, et une usine ne pompe pas forcément dans la nappe. Pour vérifier que l'absence de lien ne vient pas de ce choix d'indicateur, on utilise les volumes réellement prélevés.

La BNPE (Banque Nationale des Prélèvements quantitatifs en Eau) recense, pour chaque ouvrage (forage, puits, prise d'eau) et chaque année, le volume prélevé et son usage.

In [ ]:
# On télécharge les données sur les prélèvements (un fichier en cache par département)
url_bnpe = "https://hubeau.eaufrance.fr/api/v1/prelevements/chroniques"

def telecharger_toutes_pages(url, params_api):
    """Récupère toutes les pages d'une réponse Hub'eau en suivant le lien "next" (requete_hubeau réessaie en cas de coupure)."""
    lignes_api = []
    while url is not None:
        data = requete_hubeau(params_api, url)
        lignes_api.extend(data["data"])
        url = data.get("next")
        params_api = None   #Les paramètres sont déjà intégrés dans l'url de la page "next"
        time.sleep(0.5)     #On attend un peu pour éviter de se faire rejeter par le serveur
    return lignes_api

morceaux = []
for dep in DEPARTEMENTS:
    chemin_bnpe = f"../../data/prelevements_{dep}.csv"
    if not os.path.exists(chemin_bnpe):
        prelevements_dep = []
        #Comme pour le téléchargement des chroniques, les données de prélèvement sont trop lourdes à télécharger d'un seul coup : on les télécharge année après année.
        for annee in range(2015, 2026):
            prelevements_dep.extend(telecharger_toutes_pages(url_bnpe, {"code_departement": dep, "annee": annee, "size": 2000, "format": "json"}))
            print(f"Département {dep}, année {annee} terminée")
        pd.DataFrame(prelevements_dep).to_csv(chemin_bnpe, index=False, encoding="utf-8")
        print("Téléchargement réussi.")
    morceaux.append(pd.read_csv(chemin_bnpe, encoding="utf-8"))
prelevements = pd.concat(morceaux, ignore_index=True)

In [ ]:
url_ouvrages = "https://hubeau.eaufrance.fr/api/v1/prelevements/referentiel/ouvrages"

morceaux = []
for dep in DEPARTEMENTS:
    chemin_ouvrages = f"../../data/ouvrages_{dep}.csv"
    if not os.path.exists(chemin_ouvrages):
        pd.DataFrame(telecharger_toutes_pages(url_ouvrages, {"code_departement": dep, "size": 5000, "format": "json"})).to_csv(chemin_ouvrages, index=False, encoding="utf-8")
    morceaux.append(pd.read_csv(chemin_ouvrages, encoding="utf-8"))
ouvrages = pd.concat(morceaux, ignore_index=True).drop_duplicates(subset="code_ouvrage")

prelevements = prelevements.merge(ouvrages[["code_ouvrage", "libelle_type_milieu"]], on="code_ouvrage", how="left")
sout = prelevements[prelevements["libelle_type_milieu"].str.contains("souterrain", case=False, na=False)]
sout = gpd.GeoDataFrame(sout.drop(columns="geometry"), geometry=gpd.points_from_xy(sout["longitude"], sout["latitude"]), crs="EPSG:4326").to_crs(epsg=2154)

print(ouvrages["libelle_type_milieu"].value_counts())
print(len(prelevements), "prélèvements, dont", len(sout), "souterrains")

**Qualité des données.** Le nombre de prélèvements est stable d'une année à l'autre (500 à 614) : les données ne semblent pas tronquées. Elles s'arrêtent en 2023, en raison du délai de déclaration. Sur 776 ouvrages, 690 puisent dans une nappe et 86 en surface. On ne garde que les prélèvements souterrains, les seuls susceptibles de faire baisser une nappe.

In [ ]:
# Répartition de l'usage de l'eau pompée
noms_usages = {"AEP": "Eau potable", "IND": "Industrie", "IRR": "Irrigation"}
part = sout.groupby("code_usage")["volume"].sum().rename(index=noms_usages)
print((part / 1e6).round(1).sort_values(ascending=False))
print((100 * part / part.sum()).round(1).sort_values(ascending=False))

def fig_part_usages():
    """Part de chaque usage dans les volumes souterrains prélevés."""
    (100 * part / part.sum()).sort_values().plot(kind="barh", figsize=(8, 4), color="steelblue")
    plt.xlabel("Part des volumes souterrains prélevés (%)")
    plt.ylabel("")
    plt.title("Répartition des prélèvements en eau souterraine par usage (2015-2023)")
    plt.show()

if AFFICHER:
    fig_part_usages()

In [ ]:
# Évolution des prélèvements
par_an = sout.pivot_table(values="volume", index="annee", columns="code_usage", aggfunc="sum").rename(columns=noms_usages) / 1e6

comparaison = pd.DataFrame({"2015-2017": par_an.loc[2015:2017].mean(), "2021-2023": par_an.loc[2021:2023].mean()})
comparaison["évolution (%)"] = 100 * (comparaison["2021-2023"] / comparaison["2015-2017"] - 1)
print(comparaison.round(1))

def fig_prelevements_par_an():
    """Volumes souterrains prélevés chaque année, par usage."""
    par_an.plot(figsize=(10, 6), marker="o")
    plt.ylabel("Volume prélevé (millions de m³)")
    plt.xlabel("Année")
    plt.title("Évolution des prélèvements en eau souterraine par usage")
    plt.show()

if AFFICHER:
    fig_prelevements_par_an()

**Interprétation.** L'eau potable domine très largement les prélèvements en eau souterraine : 79,5 % des volumes entre 2015 et 2023, contre 18,2 % pour l'industrie et 2,3 % pour l'irrigation.

- **Eau potable** : environ 110 à 128 millions de m³ par an, en baisse d'environ 8 % entre 2015-2017 et 2021-2023.
- **Industrie** : stable autour de 25 à 29 millions de m³ par an (+3 % entre les deux périodes).
- **Irrigation** : volumes très faibles (2 à 7 millions de m³), plus élevés les années sèches (2020 et 2022).

L'industrie pompe donc peu dans les nappes du Nord, et de façon stable : il est cohérent que la présence d'usines ne se voie pas dans l'évolution des nappes. La corrélation légèrement positive ne s'explique pas non plus par une baisse des pompages industriels, puisque ceux-ci n'ont pas diminué.

In [ ]:
# Volume annuel moyen de chaque ouvrage souterrain, par usage
ouv = sout.groupby(["code_ouvrage", "code_usage"]).agg(volume=("volume", "mean"), geometry=("geometry", "first")).reset_index()
ouv = gpd.GeoDataFrame(ouv, geometry="geometry", crs=sout.crs).to_crs(epsg=3857)

def fig_carte_prelevements():
    """Carte des prélèvements souterrains, taille proportionnelle au volume annuel moyen."""
    couleurs = {"AEP": "royalblue", "IND": "black", "IRR": "green"}
    fig, ax = plt.subplots(figsize=(12, 10))
    for code, couleur in couleurs.items():
        sel = ouv[ouv["code_usage"] == code]
        sel.plot(ax=ax, color=couleur, markersize=np.sqrt(sel["volume"]) / 10, alpha=0.6, label=noms_usages[code], zorder=2)
    st[st["code_bss"].isin(["00143C0079/F1", "00147D0218/P1"])].to_crs(epsg=3857).plot(ax=ax, color="red", marker="*", markersize=300, label="Bondues / Lille", zorder=3)
    ctx.add_basemap(ax, source=ctx.providers.Esri.WorldTopoMap, zorder=1)
    ax.set_axis_off()
    plt.legend(loc="upper left")
    plt.title("Prélèvements souterrains annuels moyens par usage (2015-2023)")
    plt.show()

if AFFICHER:
    fig_carte_prelevements()

**Interprétation.** La taille des cercles est proportionnelle au volume annuel moyen prélevé. Les prélèvements pour l'eau potable (bleu) sont les plus importants et se concentrent autour de la métropole lilloise, de Douai-Valenciennes et de l'Avesnois. Les prélèvements industriels (noir) sont dispersés et généralement plus petits. L'irrigation (vert) correspond à de nombreux petits prélèvements en Flandre intérieure et dans le Cambrésis.

In [ ]:
# Concentrons-nous sur les prélèvements autour des stations clés (stations du Nord : ignorées si elles ne font pas partie des départements étudiés)
codes_focus = ["00143C0079/F1", "00134X0127/PZ1", "00146B0007/F1", "00146X2045/PZF", "00146B0737/PZR2", "00147X3381/PZ6", "00147D0218/P1"]
lignes = []
for _, s in st[st["code_bss"].isin(codes_focus)].iterrows():
    proches = sout[sout.distance(s.geometry) <= 5000]
    nb_annees = max(proches["annee"].nunique(), 1)
    vol = proches.groupby("code_usage")["volume"].sum() / nb_annees / 1e6
    lignes.append({
        "code_bss": s["code_bss"],
        "commune": s["nom_commune"],
        "profondeur": s["classe_profondeur"],
        "geologie": s["famille_geologique"],
        "tendance_cm_an": round(s["tendance_cm_an"], 2),
        "eau_potable": round(vol.get("AEP", 0), 2),
        "industrie": round(vol.get("IND", 0), 2),
        "total": round(vol.sum(), 2)
    })
focus = pd.DataFrame(lignes, columns=["code_bss", "commune", "profondeur", "geologie", "tendance_cm_an", "eau_potable", "industrie", "total"]).sort_values("tendance_cm_an")
print(focus.to_string(index=False))

In [ ]:
def fig_bondues_lille():
    """Trajectoires annuelles, climat retiré, de Bondues et des cinq piézomètres de Lille."""
    codes_duo = ["00143C0079/F1", "00146B0007/F1", "00146X2045/PZF", "00146B0737/PZR2", "00147X3381/PZ6", "00147D0218/P1"]
    codes_duo = [c for c in codes_duo if c in set(st["code_bss"])]   #Stations du Nord : ignorées si absentes
    plt.figure(figsize=(12, 6))
    for code in codes_duo:
        commune = st.loc[st["code_bss"] == code, "nom_commune"].iloc[0]
        style = {"color": "darkred", "linewidth": 3} if commune == "Bondues" else {"color": "steelblue", "alpha": 0.7}
        plt.plot(ecart_annuel.index, ecart_annuel[code], marker="o", label=f"{commune} ({code})", **style)
    plt.axhline(0, color="black", linewidth=0.8)
    plt.xlabel("Année")
    plt.ylabel("Écart au niveau 2015-2017, pluie retirée (m)")
    plt.title("Bondues et les cinq piézomètres de Lille, climat retiré : des pompages comparables")
    plt.legend(fontsize=8)
    plt.grid(True, alpha=0.3)
    plt.show()

if AFFICHER:
    fig_bondues_lille()

**Interprétation.** Volumes annuels moyens prélevés à moins de 5 km, en millions de m³, et tendance corrigée du climat.

- **Nieppe** (−36 cm/an) : quasiment aucun prélèvement (0,01 million de m³, uniquement pour l'irrigation). Sa baisse ne vient pas d'un pompage local, encore moins de l'industrie. C'est une nappe captive : sa tendance corrigée est à lire avec prudence.
- **Bondues** (−57 cm/an) : 5,8 millions de m³ par an, dont 36 % pour l'industrie. Le pompage y est important, mais l'eau potable y pèse davantage que l'industrie.
- **Lille** : une fois la pluie retirée, les cinq piézomètres sont presque stables, entre −4,6 et +2,7 cm/an (quatre en légère baisse), alors qu'ils sont entourés de 3,3 à 7,0 millions de m³ de prélèvements annuels. Deux d'entre eux ont plus de prélèvements totaux que Bondues, et deux autres plus de prélèvements industriels (jusqu'à 4,5 millions de m³, plus du double de Bondues).

À pompage comparable, voire supérieur, les nappes de Lille perdent au plus quelques centimètres par an, quand celle de Bondues perd plus d'un demi-mètre par an : ce n'est pas la quantité d'eau prélevée qui fait la différence.

Limite : la BNPE n'indique pas dans quelle nappe pompe chaque forage. Les prélèvements proches de Bondues ne visent pas forcément la même nappe que le piézomètre.

In [ ]:
moy = sout.groupby(["code_ouvrage", "code_usage"]).agg(volume=("volume", "mean"), geometry=("geometry", "first")).reset_index()
moy = gpd.GeoDataFrame(moy, geometry="geometry", crs=sout.crs)

st["vol_total_5km"] = st.geometry.apply(lambda g: moy.loc[moy.distance(g) <= 5000, "volume"].sum() / 1e6)
st["vol_ind_5km"] = st.geometry.apply(lambda g: moy.loc[(moy.distance(g) <= 5000) & (moy["code_usage"] == "IND"), "volume"].sum() / 1e6)

indicateurs = {
    "Nombre d'usines à 2 km": "usines_2km",
    "Nombre d'usines à 5 km": "usines_5km",
    "Nombre d'usines à 10 km": "usines_10km",
    "Volume total pompé à 5 km": "vol_total_5km",
    "Volume industriel pompé à 5 km": "vol_ind_5km"
}
#Corrélation avec la tendance corrigée du climat, et pour comparaison avec la tendance brute (avant retrait de la pluie)
tableau_corr = pd.DataFrame({
    "Indicateur de pression": list(indicateurs.keys()),
    "Spearman (tendance corrigée du climat)": [round(st["tendance_cm_an"].corr(st[c], method="spearman"), 2) for c in indicateurs.values()],
    "Spearman (tendance brute)": [round(st["pente_brute_cm_an"].corr(st[c], method="spearman"), 2) for c in indicateurs.values()]
})
print(tableau_corr.to_string(index=False))

> **À compléter après exécution.** Les corrélations avec la tendance corrigée sont-elles négatives (plus de pression, plus de baisse) ? Sont-elles fortes ou faibles ? Le résultat change-t-il selon le rayon, selon l'indicateur (usines ou volumes), et par rapport à la tendance brute ? Une corrélation ne prouve pas une cause : la géologie peut expliquer un lien apparent (les gros pompages industriels sont surtout au-dessus de la craie lilloise, qui se recharge vite).

## 5.5 Conclusion de la partie 5

**Ce que change le retrait du climat.** Dans les niveaux bruts, la nappe typique du Nord semblait stable. Une fois la pluie retirée, 41 stations sur 45 baissent légèrement (médiane −4,6 cm/an) : les années humides de la fin de période masquaient cette baisse.

**L'industrie est un préleveur minoritaire.** Elle ne représente que 18 % des prélèvements souterrains du département, contre 80 % pour l'eau potable, et ses volumes sont restés stables entre 2015 et 2023.

**À pompage comparable, des tendances très différentes.** Bondues perd plus d'un demi-mètre par an, alors que les cinq piézomètres de Lille, entourés de prélèvements comparables ou supérieurs, sont presque stables. La quantité d'eau prélevée à proximité ne suffit donc pas à expliquer les fortes baisses.

> **À compléter après exécution.** Le lien entre la tendance corrigée et la pression industrielle (groupes et corrélations des 5.2 et 5.4), le rôle de la géologie (5.3), et la réponse à la problématique.

### Limites

- Un échantillon réduit : 45 piézomètres bien suivis, avec de petits effectifs dans certains groupes.
- Les plus fortes baisses sont des nappes captives, pour lesquelles le modèle climat est à lire avec prudence.
- Le modèle climat compte comme « climat » ce que les prélèvements ont de saisonnier ou de lié à la sécheresse : la tendance corrigée sous-estime probablement l'effet humain.
- Des corrélations, qui ne prouvent pas une causalité. La géologie et la profondeur se recoupent en partie.

### Perspective

Il serait intéressant de reproduire cette analyse dans une région plus sèche, ou dont l'industrie est plus consommatrice d'eau, pour voir si les conclusions restent les mêmes.

# Conclusion générale

**La géographie fixe le niveau des nappes, pas leurs variations.** Le niveau moyen d'une nappe suit l'altitude de sa station, sauf pour quatre nappes captives rabattues sous le niveau de la mer. L'ampleur des variations, elle, ne s'explique ni par l'altitude, ni par la roche, ni par la profondeur, dès que l'on met à part les piézomètres superficiels de la Scarpe (partie 3).

**Le climat pilote l'essentiel des variations, et masque une légère baisse.** Le cycle saisonnier de chaque nappe varie d'une année à l'autre avec la pluie de la saison (partie 4). Une fois la part de la pluie retirée, 41 stations sur 45 baissent, avec une médiane de −4,6 cm/an : les années humides de la fin de période masquaient cette baisse dans le niveau observé (parties 4 et 5).

**L'industrie est un préleveur minoritaire.** Elle ne représente que 18 % des prélèvements souterrains, contre 80 % pour l'eau potable, et ses volumes sont restés stables. À pompage comparable, Bondues perd plus d'un demi-mètre par an quand les piézomètres de Lille sont presque stables (partie 5).

> **À compléter après exécution.** Le lien entre la tendance corrigée du climat et la pression industrielle (partie 5), puis la réponse à la problématique : l'exposition locale aux activités industrielles amplifie-t-elle la baisse des nappes, et quel conseil en tirer pour l'implantation de nouvelles usines très consommatrices d'eau ?

**Limites.**
- Un échantillon réduit (45 piézomètres bien suivis), avec de petits effectifs dans certains groupes ; la géologie et la profondeur se recoupent.
- Des corrélations, qui ne prouvent pas une causalité.
- Le modèle climat sous-estime probablement l'effet humain : ce que les prélèvements ont de saisonnier ou de lié à la sécheresse est compté comme « climat ». Les plus fortes baisses sont des nappes captives, pour lesquelles il est à lire avec prudence.
- La BNPE n'indique pas dans quelle nappe pompe chaque forage.

**Perspective.** Reproduire l'analyse dans une région plus sèche ou dont l'industrie consomme davantage d'eau.